# Parsing `ingr_map.pkl`

`ingr_map.pkl` adalah tabel mapping dari teks bahan mentah (raw ingredient) → versi yang sudah dinormalisasi (mis. `replaced`) dan `id` ingredient. Biasanya dipakai untuk:
- menstandarkan berbagai variasi penulisan bahan ("romaine lettuce leaf", "mixed baby lettuces", dll)
- mengubah daftar ingredient pada resep menjadi `ingredient_id` konsisten untuk modeling / rekomendasi

In [3]:
import ast
import re
import json

from pathlib import Path

import pandas as pd

In [2]:
pkl_path = Path("dataset/ingr_map.pkl")
df = pd.read_pickle(pkl_path)

print("Loaded:", pkl_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print("Unique ingredient ids:", df["id"].nunique())

display(df.head(10))

# Mapping cepat untuk lookup
processed_map = (
    df.sort_values("count", ascending=False)
    .drop_duplicates("processed")
    .set_index("processed")[["id", "replaced", "count"]]
)

id_to_name = (
    df.sort_values("count", ascending=False)
    .drop_duplicates("id")
    .set_index("id")["replaced"]
)

Loaded: dataset\ingr_map.pkl
Shape: (11659, 7)
Columns: ['raw_ingr', 'raw_words', 'processed', 'len_proc', 'replaced', 'count', 'id']
Unique ingredient ids: 8023


,raw_ingr,raw_words,processed,len_proc,replaced,count,id
0,"medium heads bibb or red leaf lettuce, washed,...",13,"medium heads bibb or red leaf lettuce, washed,...",73,lettuce,4507,4308
1,mixed baby lettuces and spring greens,6,mixed baby lettuces and spring green,36,lettuce,4507,4308
2,romaine lettuce leaf,3,romaine lettuce leaf,20,lettuce,4507,4308
3,iceberg lettuce leaf,3,iceberg lettuce leaf,20,lettuce,4507,4308
4,red romaine lettuce,3,red romaine lettuce,19,lettuce,4507,4308
5,head romaine lettuce,3,head romaine lettuce,20,lettuce,4507,4308
6,curly endive lettuce,3,curly endive lettuce,20,lettuce,4507,4308
7,romaine lettuce hearts,3,romaine lettuce heart,21,lettuce,4507,4308
8,baby leaf lettuce,3,baby leaf lettuce,17,lettuce,4507,4308
9,head of lettuce,3,head of lettuce,15,lettuce,4507,4308


In [4]:
def parse_str_list(x):
    """Parse kolom `ingredients` menjadi list[str]."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            value = ast.literal_eval(s)
            return value if isinstance(value, list) else []
        except Exception:
            return []
    return [s]


def parse_nutrition(x):
    """Parse kolom `nutrition` menjadi list[float]."""
    if isinstance(x, (list, tuple)):
        return [float(v) for v in x]
    if pd.isna(x):
        return []
    s = str(x).strip()
    if not s:
        return []
    if s.startswith("[") and s.endswith("]"):
        try:
            value = ast.literal_eval(s)
            if isinstance(value, (list, tuple)):
                return [float(v) for v in value]
        except Exception:
            return []
    return []


# Nutrition names pada dataset
NUTRITION_COLS = [
    "calories",
    "total_fat_pdv",
    "sugar_pdv",
    "sodium_pdv",
    "protein_pdv",
    "saturated_fat_pdv",
    "carbohydrates_pdv",
]


def split_nutrition_column(recipes_df: pd.DataFrame, nutrition_col: str = "nutrition") -> pd.DataFrame:
    """Expand `nutrition` list ke kolom numerik. Kalau format tidak sesuai, kolom akan NaN."""
    nutrition_lists = recipes_df[nutrition_col].apply(parse_nutrition)
    for i, name in enumerate(NUTRITION_COLS):
        recipes_df[name] = nutrition_lists.apply(lambda lst: lst[i] if len(lst) > i else pd.NA)
        recipes_df[name] = pd.to_numeric(recipes_df[name], errors="coerce")
    return recipes_df


def _normalize_key(text: str) -> str:
    """Normalisasi sederhana: lowercase, trim, ganti whitespace berlebih dengan spasi tunggal."""
    return re.sub(r"\s+", " ", str(text).lower().strip())


def _simple_variants(key: str) -> list[str]:
    """Heuristik kecil untuk variasi umum (plural, spasi)."""
    variants = [key]
    if key.endswith("es") and len(key) > 3:
        variants.append(key[:-2])
    if key.endswith("s") and len(key) > 2:
        variants.append(key[:-1])

    seen = set()
    out = []
    for v in variants:
        if v and v not in seen:
            seen.add(v)
            out.append(v)
    return out


def build_processed_lookup(processed_map_df: pd.DataFrame):
    """Buat dict lookup cepat dari `processed_map`"""
    processed_to_id = processed_map_df["id"].to_dict()
    processed_to_name = processed_map_df["replaced"].to_dict()
    return processed_to_id, processed_to_name


def map_ingredients_to_ids(ingredients_list: list, processed_to_id: dict, processed_to_name: dict):
    """Map list ingredient strings → (ids, names, unmapped)."""
    ids = []
    names = []
    unmapped = []
    for ingr in ingredients_list:
        key = _normalize_key(ingr)
        ingr_id = None
        ingr_name = None
        for variant in _simple_variants(key):
            if variant in processed_to_id:
                raw_id = processed_to_id.get(variant)
                raw_name = processed_to_name.get(variant)
                ingr_id = int(raw_id) if pd.notna(raw_id) else None
                ingr_name = str(raw_name) if pd.notna(raw_name) else None
                break
        if ingr_id is None:
            unmapped.append(str(ingr))
        else:
            ids.append(ingr_id)
            names.append(ingr_name)
    return ids, names, unmapped

## Cleansing `RAW_recipes.csv`

In [6]:
raw_recipes_path = Path("dataset/RAW_recipes.csv")
out_csv_path = Path("dataset/RAW_recipes_cleaned.csv")

if not raw_recipes_path.exists():
    raise FileNotFoundError(raw_recipes_path)

processed_to_id, processed_to_name = build_processed_lookup(processed_map)

# Memakai chunking karena filenya besar
CHUNKSIZE = 50_000

def cleanse_recipes_chunk(chunk: pd.DataFrame) -> pd.DataFrame:
    # nutrition -> kolom numerik
    chunk = split_nutrition_column(chunk, nutrition_col="nutrition")

    # ingredients -> list
    chunk["ingredients_list"] = chunk["ingredients"].apply(parse_str_list)

    # map ingredients -> ids/names/unmapped
    mapped = chunk["ingredients_list"].apply(
        lambda lst: map_ingredients_to_ids(lst, processed_to_id, processed_to_name)
    )
    chunk["ingredient_ids"] = mapped.apply(lambda t: t[0])
    chunk["ingredient_names"] = mapped.apply(lambda t: t[1])
    chunk["ingredients_unmapped"] = mapped.apply(lambda t: t[2])

    # Simpan list sebagai JSON string agar bisa jadi "array" lagi via json.loads
    chunk["ingredient_ids"] = chunk["ingredient_ids"].apply(json.dumps)
    chunk["ingredient_names"] = chunk["ingredient_names"].apply(json.dumps)
    chunk["ingredients_unmapped"] = chunk["ingredients_unmapped"].apply(json.dumps)

    # DROP kolom mentah yang sudah diproses
    drop_cols = [c for c in ["nutrition", "ingredients", "ingredients_list"] if c in chunk.columns]
    chunk = chunk.drop(columns=drop_cols)
    return chunk

# Jalankan cleansing dan simpan ke CSV
first = True
total_rows = 0
for chunk in pd.read_csv(raw_recipes_path, chunksize=CHUNKSIZE):
    cleaned = cleanse_recipes_chunk(chunk)
    cleaned.to_csv(out_csv_path, mode="w" if first else "a", index=False, header=first)
    total_rows += len(cleaned)
    first = False
    print("Processed rows:", total_rows)

print("Saved cleaned file ->", out_csv_path)

Processed rows: 50000
Processed rows: 100000
Processed rows: 150000
Processed rows: 200000
Processed rows: 231637
Saved cleaned file -> dataset\RAW_recipes_cleaned.csv
